# Urban100 round 2: whole image versus 512-px tiles for the one-step ADM posterior mean (fused attention)

Round 1 (`urban100-one-step`) passed 3-D q, k, v to `scaled_dot_product_attention`. The fused kernels need 4-D
inputs, so it fell back to the math kernel, materialised the N² attention matrix and ran out of memory on the
larger images. On the dev images that did fit, 512-px tiles (overlap 128) beat the whole image by
+0.07 / +0.10 / +0.17 dB (10 of 10 images), which reopened the image-size hypothesis.

This run uses 4-D q, k, v (fused kernels, checked against ADM's legacy attention) on all 100 images × σ 15/25/50:
* whole image ×1 / ×2 (horizontal flip) and 512-px tiles ×1 / ×2;
* on dev-12 only, for later choices: 384- and 768-px tiles, and attention logits scaled by sqrt(log N / log n_train).

Pre-registered (amendment 1, before any confirm-88 tile score): T2 confirmed if tiles-512 ×2 − whole ×2 is
≥ +0.05 dB at every σ on confirm-88 with Holm sign-flip p < 0.01 and ≥ 65 % of images better; killed if < +0.02 at
every σ. Pilots (both ×2 outputs, float16) are saved for the CPU non-local stage.
Noise: DMID's protocol with the seeds of `diffusion-tweedie-benchmark` (Urban100 is the fourth set).
Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

SIGMAS = [15, 25, 50]
SEED_SETS = ["McMaster", "Kodak24", "CBSD68", "Urban100"]
DEV12 = list(range(8, 100, 8))
TILES = {512: 128, 384: 96, 768: 192}       # tile: overlap; 512 is the pre-registered setting
TIME_LIMIT_H = 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, f"{OUT}/pilots"):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)


def download(url, path, size=None):
    """Resumable download; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


t0 = time.time()
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": ("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", 367_018_356),
}
with ThreadPoolExecutor(2) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
with zipfile.ZipFile(P["datasets"]) as z:
    names = sorted([n for n in z.namelist() if n.startswith("Datasets/Gaussion/Urban100/") and not n.endswith("/")], key=natkey)
    TEST = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
assert len(TEST) == 100
TIMES["downloads"] = time.time() - t0
print(len(TEST), "images", f"downloads {TIMES['downloads'] / 60:.1f} min")


def noisy_of(clean_u8, sigma, idx):
    """DMID's protocol: sample std exactly sigma/255, unclipped; seeds of diffusion-tweedie-benchmark."""
    seed = 1_000_000 * (SEED_SETS.index("Urban100") + 1) + 1000 * sigma + idx
    x = torch.from_numpy(clean_u8).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(a.shape[0], 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=a.shape[0])[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def to_u8(x):
    return (x[0].clamp(0, 1) * 255).round()


def scores(out_u8, ref_u8, lp):
    a, b = out_u8.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    with torch.no_grad():
        lpv = lp(a.float()[None] / 127.5 - 1, b.float()[None] / 127.5 - 1).item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf"), ssim=ssim(a, b), lpips=lpv)

## 1 · ADM with 4-D q, k, v (fused SDPA kernels; optional logit scaling), checked against the legacy attention

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet

LEGACY_ATTENTION = gd_unet.QKVAttentionLegacy.forward


def sdpa_attention(self, qkv):
    """ADM's legacy attention through SDPA with 4-D (batch, heads, tokens, channels) inputs, so the fused kernels apply.
    If the module carries `train_area_ratio` (256² over the padded input area), logits are scaled by
    sqrt(log N / log n_train), with n_train = N · ratio the token count at 256² input."""
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.reshape(bs, self.n_heads, ch, length).transpose(2, 3).contiguous()
               for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    ratio, scale = getattr(self, "train_area_ratio", None), None
    if ratio is not None and ratio < 1 and length * ratio > 1:
        scale = math.sqrt(math.log(length) / math.log(length * ratio)) / math.sqrt(ch)
    return F.scaled_dot_product_attention(q, k, v, scale=scale).transpose(2, 3).reshape(bs, -1, length)


ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_adm(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                     use_scale_shift_norm=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False).convert_to_fp16()
    return m


import lpips

CTX = [dict(dev=d, adm=make_adm(d), lpips=lpips.LPIPS(net="alex", verbose=False).to(d)) for d in DEVICES]
ADM_STATE.clear()
with torch.no_grad():  # the 4-D SDPA attention equals ADM's legacy attention
    probe, tt = torch.randn(2, 3, 128, 128, device=DEVICES[0]), torch.tensor([30.0, 300.0], device=DEVICES[0])
    gd_unet.QKVAttentionLegacy.forward = LEGACY_ATTENTION
    ref = CTX[0]["adm"](probe, tt).float()
    gd_unet.QKVAttentionLegacy.forward = sdpa_attention
    ATTN_GAP = float((CTX[0]["adm"](probe, tt).float() - ref).abs().max())
print(f"max |SDPA − legacy attention| on ADM's output: {ATTN_GAP:.2e}")
assert ATTN_GAP < 5e-2

## 2 · Estimators

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


@torch.no_grad()
def tweedie(model, y, t, ab):
    """One-step posterior mean of y [B,3,H,W] ([0,1] units, unclipped) at continuous step t with abar ab."""
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3].float()
    return (((x - math.sqrt(1 - ab) * eps) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


def tiled(fn, y, tile, overlap, batch=2):
    """fn applied to overlapping tiles of y, blended with linear ramps."""
    _, _, H, W = y.shape
    stride = tile - overlap
    cover = lambda n: max(tile, tile + math.ceil(max(n - tile, 0) / stride) * stride)
    yp = F.pad(y, (0, cover(W) - W, 0, cover(H) - H), mode="reflect")
    r = torch.arange(tile, dtype=torch.float32, device=y.device)
    ramp = torch.clamp(torch.minimum(r + 1, tile - r) / overlap, max=1)
    w2 = ramp[:, None] * ramp[None, :]
    acc, wsum = torch.zeros_like(yp), torch.zeros_like(yp[:, :1])
    coords = [(i, j) for i in range(0, yp.shape[-2] - tile + 1, stride) for j in range(0, yp.shape[-1] - tile + 1, stride)]
    for b in range(0, len(coords), batch):
        cs = coords[b:b + batch]
        res = fn(torch.cat([yp[..., i:i + tile, j:j + tile] for i, j in cs]))
        for (i, j), out in zip(cs, res):
            acc[..., i:i + tile, j:j + tile] += out * w2
            wsum[..., i:i + tile, j:j + tile] += w2
    return (acc / wsum)[..., :H, :W]


def with_kappa(model, y, fn):
    """Run fn() with attention-logit scaling set for y's padded size on every attention module of model."""
    hp, wp = (-(-y.shape[-2] // 64)) * 64, (-(-y.shape[-1] // 64)) * 64
    mods = [m for m in model.modules() if isinstance(m, gd_unet.QKVAttentionLegacy)]
    for m in mods:
        m.train_area_ratio = 256 * 256 / (hp * wp)
    try:
        return fn()
    finally:
        for m in mods:
            del m.train_area_ratio

## 3 · Every estimator on every image and σ (each one guarded); both ×2 outputs saved as float16 pilots

In [ ]:
def run_image(ctx, job):
    idx, sigma = job
    name, clean = TEST[idx]
    dev, adm, lp = ctx["dev"], ctx["adm"], ctx["lpips"]
    split = "dev" if idx in DEV12 else "confirm"
    y = noisy_of(clean, sigma, idx)
    info = dict(image=name, idx=idx, sigma=sigma, split=split, height=clean.shape[0], width=clean.shape[1],
                noisy_sum=float(y.double().sum()))
    y = y[None].to(dev)
    ref = torch.from_numpy(clean).permute(2, 0, 1).to(dev).float()
    t, ab = matched(sigma / 255)
    outs, errors = {}, []

    def timed(label, fn):
        try:
            t0 = time.time()
            outs[label] = fn()
            torch.cuda.synchronize(dev)
            info[f"seconds {label}"] = time.time() - t0
        except Exception:
            errors.append((label, traceback.format_exc(limit=3)))
            torch.cuda.empty_cache()

    one = lambda z: tweedie(adm, z, t, ab)
    timed("whole ×1", lambda: one(y))
    timed("whole flipped", lambda: one(y.flip(-1)).flip(-1))
    timed("tiles-512 ×1", lambda: tiled(one, y, 512, TILES[512]))
    timed("tiles-512 flipped", lambda: tiled(one, y.flip(-1), 512, TILES[512]).flip(-1))
    for kind in ("whole", "tiles-512"):
        if f"{kind} ×1" in outs and f"{kind} flipped" in outs:
            outs[f"{kind} ×2"] = (outs[f"{kind} ×1"] + outs.pop(f"{kind} flipped")) / 2
            np.savez_compressed(f"{OUT}/pilots/{kind}_s{sigma}_{idx:03d}.npz", x=outs[f"{kind} ×2"][0].cpu().numpy().astype(np.float16))
        outs.pop(f"{kind} flipped", None)
    if split == "dev":
        for tile in (384, 768):
            timed(f"tiles-{tile} ×1", lambda tile=tile: tiled(one, y, tile, TILES[tile], batch=1 if tile > 512 else 2))
        timed("whole ×1, attention κ", lambda: with_kappa(adm, y, lambda: one(y)))
    info["peak_gib"] = torch.cuda.max_memory_allocated(dev) / 2 ** 30
    rows = [dict(image=name, idx=idx, sigma=sigma, split=split, method=m, **scores(to_u8(o), ref, lp)) for m, o in outs.items()]
    return rows, info, errors


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, failures)."""
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(20)
        if time.time() - last > 600:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(failures)} failed, {(time.time() - t0) / 60:.1f} min", flush=True)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
by_area = sorted(range(len(TEST)), key=lambda i: -TEST[i][1].shape[0] * TEST[i][1].shape[1])
smoke = [(by_area[0], 50), (DEV12[0], 25)]   # the largest image first: memory check before the full run
res, fail = worker_pool(smoke, run_image)
print("smoke test:", len(res), "ok,", len(fail), "failed;", [e[0] for _, _, errs in res for e in errs],
      [{k: round(v, 2) for k, v in info.items() if k.startswith(("seconds", "peak"))} for _, info, _ in res])
assert not fail and not any(errs for _, _, errs in res), "smoke test failed"
jobs = [(i, s) for s in SIGMAS for i in range(len(TEST)) if (i, s) not in smoke]
res2, fail2 = worker_pool(jobs, run_image)
res, fail = res + res2, fail + fail2
SCORES = pd.DataFrame([r for rows, _, _ in res for r in rows])
INFO = pd.DataFrame([info for _, info, _ in res])
METHOD_ERRORS = [(info["idx"], info["sigma"], label, tb) for _, info, errs in res for label, tb in errs]
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
INFO.to_csv(f"{OUT}/image_info.csv", index=False)
TIMES["denoising"] = time.time() - t0
print(f"{len(res)}/{len(smoke) + len(jobs)} jobs, {len(fail)} failed jobs, {len(METHOD_ERRORS)} failed methods, {TIMES['denoising'] / 60:.1f} min")
for e in METHOD_ERRORS[:3]:
    print("METHOD ERROR", e)

## 4 · Results and the pre-registered T2 test (confirm-88)

In [ ]:
def sign_flip_p(d, n_flip=200_000, seed=0):
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1.0, 1.0], size=(n_flip, len(d)))
    return float((np.abs((signs * d).mean(1)) >= abs(d.mean()) - 1e-12).mean())


mean = SCORES.groupby(["method", "sigma"])[["psnr", "ssim", "lpips"]].mean()
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()}, all images run")
    print(mean[metric].unstack().round(3 if metric == "psnr" else 4).to_string())
print("\nmean PSNR, dev-12")
print(SCORES[SCORES.split == "dev"].groupby(["method", "sigma"]).psnr.mean().unstack().round(3).to_string())
print("\nreference PSNR (all 100): DMID released 35.26 / 33.11 / 30.28; GRL-B 35.54 / 33.35 / 30.46; Hi-IR 35.46 / 33.34 / 30.59;"
      " MaIR+ 35.42 / 33.30 / 30.41; our Restormer rerun 35.133 / 32.954 / 30.018")
wide = SCORES.pivot_table(index=["idx", "sigma"], columns="method", values="psnr")
rows = []
for s in SIGMAS:
    sub = wide.xs(s, level="sigma")
    conf = sub[[i not in DEV12 for i in sub.index]]
    for a, b, part in [("tiles-512 ×2", "whole ×2", conf), ("whole ×2", "whole ×1", conf), ("tiles-512 ×2", "tiles-512 ×1", conf)]:
        d = (part[a] - part[b]).dropna().to_numpy()
        rows.append(dict(contrast=f"{a} − {b}", sigma=s, n=len(d), mean=d.mean(), ci95=1.96 * d.std(ddof=1) / math.sqrt(len(d)),
                         better=float((d > 0).mean()), p=sign_flip_p(d)))
PAIRED = pd.DataFrame(rows)
t2 = PAIRED[PAIRED.contrast == "tiles-512 ×2 − whole ×2"].sort_values("p").copy()
t2["p_holm"] = np.minimum(1.0, np.maximum.accumulate(t2.p.to_numpy() * (len(t2) - np.arange(len(t2)))))
PAIRED = PAIRED.merge(t2[["contrast", "sigma", "p_holm"]], on=["contrast", "sigma"], how="left")
print("\npaired PSNR differences on confirm-88 (dB)")
print(PAIRED.round(4).to_string(index=False))
T2 = PAIRED[PAIRED.contrast == "tiles-512 ×2 − whole ×2"]
t2_success = bool((T2["mean"] >= 0.05).all() and (T2.p_holm < 0.01).all() and (T2.better >= 0.65).all())
t2_kill = bool((T2["mean"] < 0.02).all())
print(f"\nT2 (pre-registered): success {t2_success}, kill {t2_kill}")
PAIRED.to_csv(f"{OUT}/paired.csv", index=False)
print(INFO[[c for c in INFO if c.startswith(("seconds", "peak"))]].describe().round(2).to_string())
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, attention_gap=ATTN_GAP, means=mean.reset_index().to_dict("records"),
                   paired=PAIRED.to_dict("records"), t2_success=t2_success, t2_kill=t2_kill, dev12=DEV12,
                   failed_jobs=[j for j, _ in fail], failed_methods=[e[:3] for e in METHOD_ERRORS]), f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(len(os.listdir(f"{OUT}/pilots")), "pilots saved", f"total {(time.time() - T_START) / 60:.1f} min")